# Nuclei Fitting

Try to fit a line to hyphal nuclei to reconstruct the hyphal structure

In [ ]:
%load_ext autoreload
%autoreload 1
%aimport image_processing_tools

## Fit spline

Deviates from the hyphal structure a lot, bad

In [ ]:
# from image_processing_tools.util.load_files import find_files_by_pattern
# from image_processing_tools.image_class.image_container import ImageContainer
# from pathlib import Path
# import matplotlib
# # matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

# search_path = ("~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",)
# file_pattern = ("CROP_*",
#                 "MAX_*",
#                 "SHARPEST*",)

# config = {
#     "preprocessing": {
#         "resize_image": False,
#         "max_dim": 1080,
#         "outlier_percentile": 0.35,
#         "quantization": "8bit"
#     }
# }

# # Find the files. The 'files' variable will hold the list of Path objects.
# files = find_files_by_pattern(search_path[0], file_pattern[0], verbose=True)

# # Create output directory
# base_input_dir = Path("~/A8/Data_Roan/").expanduser()
# base_output_dir = Path("output/fit_hypha")

# # Assuming 'files' is defined in your notebook environment as a list of Path objects
# relative_dir = files[0].parent.relative_to(base_input_dir)
# current_output_dir = base_output_dir / relative_dir
# current_output_dir.mkdir(parents=True, exist_ok=True)

# print(f"The current output dir is {current_output_dir}")

In [ ]:
# from image_processing_tools.image_class.image_filters import ImageJProcessor
# import matplotlib.pyplot as plt

# img_c = ImageContainer(files[0],config)
# pipeline = ImageJProcessor(img_c.merge())
# pipeline.reset()
# img_g2 = pipeline.fft_bandpass_filter()
# plt.imshow(img_g2)

In [ ]:
# import cv2
# import numpy as np
# import matplotlib.pyplot as plt
# from scipy import ndimage
# from scipy.interpolate import splprep, splev
# from skimage.filters import threshold_otsu
# from skimage.measure import label, regionprops
# from skimage.segmentation import watershed
# from skimage.feature import peak_local_max
# from skimage.color import label2rgb

# def extract_and_plot_nuclei_axis(image_source):
#     """
#     Extracts nuclei using Otsu thresholding and watershed separation,
#     fits ellipses to them, plots the major axis, and fits a spline 
#     through the nuclei centroids on the original image.
    
#     Args:
#         image_source (str or np.ndarray): Path to the image file or a numpy array.
#     """
#     # 1. Load and Preprocess Image
#     if isinstance(image_source, str):
#         image = cv2.imread(image_source, cv2.IMREAD_GRAYSCALE)
#         if image is None:
#             raise FileNotFoundError(f"Could not load image from {image_source}")
#     elif isinstance(image_source, np.ndarray):
#         if image_source.ndim == 3:
#             image = cv2.cvtColor(image_source, cv2.COLOR_BGR2GRAY)
#         else:
#             image = image_source
#     else:
#         raise TypeError("Input must be a file path or a numpy array.")

#     # 2. Otsu Thresholding
#     thresh_val = threshold_otsu(image)
#     binary_mask = image > thresh_val

#     # 3. Separate Touching Nuclei (Watershed)
#     binary_mask_filled = ndimage.binary_fill_holes(binary_mask)
#     distance = ndimage.distance_transform_edt(binary_mask_filled)
    
#     # Smooth distance map to prevent over-segmentation
#     distance_smoothed = ndimage.gaussian_filter(distance, sigma=2)
    
#     coords = peak_local_max(distance_smoothed, min_distance=5, labels=binary_mask_filled)
#     mask = np.zeros(distance.shape, dtype=bool)
#     mask[tuple(coords.T)] = True
#     markers, _ = ndimage.label(mask)
    
#     labels = watershed(-distance, markers, mask=binary_mask_filled)

#     # 4. Regionprops
#     props = regionprops(labels)

#     # 5. Plotting Setup
#     fig, axes = plt.subplots(1, 3, figsize=(18, 6))
#     ax_orig = axes[0]
#     ax_mask = axes[1]
#     ax_overlay = axes[2]

#     ax_orig.imshow(image, cmap='gray')
#     ax_orig.set_title('Original Image')
#     ax_orig.axis('off')

#     colored_labels = label2rgb(labels, bg_label=0, bg_color=(0, 0, 0))
#     ax_mask.imshow(colored_labels)
#     ax_mask.set_title('Individual Nuclei (Colored)')
#     ax_mask.axis('off')

#     ax_overlay.imshow(image, cmap='gray')
#     ax_overlay.set_title('Nuclei Axes & Spline Fit')
#     ax_overlay.axis('off')

#     # 6. Process Nuclei and Collect Centroids
#     valid_centroids = [] # List to store (y, x) tuples

#     for prop in props:
#         if prop.area < 10: 
#             continue

#         y0, x0 = prop.centroid
#         valid_centroids.append((y0, x0))

#         orientation = prop.orientation
#         major_axis = prop.major_axis_length
        
#         x_offset = (major_axis / 2) * np.sin(orientation)
#         y_offset = (major_axis / 2) * np.cos(orientation)
        
#         x1 = x0 - x_offset
#         y1 = y0 - y_offset
#         x2 = x0 + x_offset
#         y2 = y0 + y_offset

#         ax_overlay.plot((x1, x2), (y1, y2), '-r', linewidth=1.5)
#         ax_overlay.plot(x0, y0, '.g', markersize=5)

#     # 7. Fit Spline Through Centroids
#     valid_centroids = np.array(valid_centroids)
    
#     # We need at least 3 points to fit a spline (default k=3)
#     if len(valid_centroids) >= 3:
#         try:
#             # Extract coordinates (y is row, x is col)
#             y = valid_centroids[:, 0]
#             x = valid_centroids[:, 1]

#             # --- Sort Points Spatially ---
#             # To fit a clean spline, points must be ordered. We use PCA to find the 
#             # principal axis of the nuclei distribution and sort points along it.
            
#             # Center the data
#             mx, my = np.mean(x), np.mean(y)
#             data_centered = np.vstack((x - mx, y - my)).T
            
#             # Compute Covariance and Eigenvectors
#             cov = np.cov(data_centered.T)
#             evals, evecs = np.linalg.eig(cov)
            
#             # Get the eigenvector corresponding to the largest eigenvalue (main direction)
#             main_axis_idx = np.argmax(evals)
#             main_axis = evecs[:, main_axis_idx]
            
#             # Project points onto the main axis
#             projections = data_centered @ main_axis
            
#             # Sort indices based on projection value
#             sort_order = np.argsort(projections)
            
#             x_sorted = x[sort_order]
#             y_sorted = y[sort_order]

#             # --- Fit B-Spline ---
#             # s=0 forces the spline to pass through all points (interpolation)
#             # k is the degree of the spline (usually 3 for cubic)
#             k_degree = min(3, len(valid_centroids) - 1)
#             tck, u = splprep([x_sorted, y_sorted], s=0, k=k_degree)
            
#             # Generate smooth curve points
#             u_new = np.linspace(0, 1, 200)
#             x_spline, y_spline = splev(u_new, tck)
            
#             # Plot Spline
#             ax_overlay.plot(x_spline, y_spline, 'c--', linewidth=2, label='Spline Fit')
#             ax_overlay.legend(loc='upper right')
            
#         except Exception as e:
#             print(f"Spline fitting failed: {e}")

#     plt.tight_layout()
#     plt.show()
    
#     return markers


# markers = extract_and_plot_nuclei_axis(img_g2)
# # np.unique(markers)

## Network Approach

### Simple case

#### Load data

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = ("~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",
               "~/A8/Data_Roan/260228_She3_Mutants_DAPI",)
file_pattern = ("CROP_*",
                "MAX_*",
                "SHARPEST*",
                "MAX_CET145*")

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[0], file_pattern[0], verbose=True)

# Create output directory
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/fit_hypha")

# Assuming 'files' is defined in your notebook environment as a list of Path objects
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

print(f"The current output dir is {current_output_dir}")

In [ ]:
from image_processing_tools.image_class.image_filters import ImageJProcessor
import matplotlib.pyplot as plt

crop_img = ImageContainer(files[0],config).merge()
pipeline_seg = ImageJProcessor(crop_img)
pipeline_seg.reset()
pipeline_seg.fft_bandpass_filter()
pipeline_seg.enhance_contrast_clahe(slope=1.5)
pipeline_seg.imagej_rolling_ball(radius=60)
pipeline_seg.gaussian_blur(sigma=3)
seg_img = pipeline_seg.return_image()

pipeline_intens = ImageJProcessor(crop_img)
pipeline_intens.reset()
pipeline_intens.fft_bandpass_filter()
pipeline_intens.enhance_contrast_clahe(slope=1)
pipeline_intens.gaussian_blur(sigma=5)
intens_img = pipeline_intens.return_image()

fig, axes = plt.subplots(1,2,figsize=(8,4))
axes[0].imshow(seg_img, cmap='gray'); axes[0].set_title('Image for Segmentation')
axes[1].imshow(intens_img, cmap='gray'); axes[1].set_title('Image for Intensity')

for ax in axes: ax.axis('off')
# plt.axis('off')
plt.tight_layout()
plt.show()

#### Extract nuclei and calculate scores

In [ ]:
from image_processing_tools.dapi_tracing.deprecated.greedy_connectivity import extract_and_plot_nuclei_axis
from pathlib import Path

adj_mat, cents, ax_lengths = extract_and_plot_nuclei_axis(segmentation_source=crop_img,
                             intensity_source=intens_img,
                             use_watershed=False,
                             path_width=15,
                             lower_size_factor=0.33, upper_size_factor=8.0,
                             use_orientation_penalty=True,
                             min_eccentricity = 0.2,
                             model=Path('~/Projects/C_Albicans Thesis Project/8. Code/notebooks/2. Data Processing/rf_models/nuclei_trace_bg_20260228DAPI.joblib'))

In [ ]:
adj_mat[adj_mat<0.2]=0
print(adj_mat)


In [ ]:
print(np.array(cents))

In [ ]:
print(ax_lengths)

#### Trace hypha and plot

In [ ]:
from image_processing_tools.dapi_tracing.deprecated.hyphal_tracer import *

# Usage:
tracer = HyphalTracer(adj_mat,cents)
valid_hyphae, rejected = tracer.trace_hyphae(min_length=2, linearity_threshold=0.5,connectivity_threshold=0.1)
plot_hyphal_reconstruction(crop_img, valid_hyphae, rejected)

#### Distance between nuclei vs. avg. nuclei length

In [ ]:


global_ratio, per_hypha_ratios = calculate_hyphal_spacing_ratio(valid_hyphae, cents, ax_lengths, )

In [ ]:
print(global_ratio,per_hypha_ratios)

### More complex case

When there are epithelial and hyphal nuclei at the same time

In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = ("~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",)
file_pattern = ("CROP_*",
                "MAX_*",
                "SHARPEST*",
                "C[0-9]*")

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[0], file_pattern[2], verbose=True)

dapi = ImageContainer(files[1],config)
# Create output directory
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/fit_hypha")

# Assuming 'files' is defined in your notebook environment as a list of Path objects
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

print(f"The current output dir is {current_output_dir}")

In [ ]:
from image_processing_tools.image_class.image_filters import ImageJProcessor
from skimage.filters import threshold_otsu, threshold_local
import matplotlib.pyplot as plt

img = dapi.merge()
pipeline = ImageJProcessor(img)
pipeline.reset()
pipeline.fft_bandpass_filter(40,2)
pipeline.enhance_contrast_clahe(block_size=18, slope=6)
pipeline.imagej_rolling_ball(radius=80, use_paraboloid=False)
# pipeline.gaussian_blur(sigma=3)
img_processed = pipeline.return_image()
thresh_val = threshold_otsu(img_processed)
local_thresh = threshold_local(img_processed, block_size=127,offset=0)
global_threshed = img_processed > thresh_val
local_threshed = img_processed > local_thresh
img_binary = global_threshed & local_threshed

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Plotting
axes[0, 0].imshow(img, cmap='gray')
axes[0, 0].set_title('DAPI Original')

axes[0, 1].imshow(img_processed, cmap='gray')
axes[0, 1].set_title('Processed')

axes[0, 2].imshow(global_threshed, cmap='viridis')
axes[0, 2].set_title('Global')

axes[1, 0].imshow(local_threshed, cmap='viridis')
axes[1, 0].set_title('Local')

axes[1, 1].imshow(img_binary, cmap='viridis')
axes[1, 1].set_title('Global & Local')

# Iterate over all axes (flattened) to turn them off
# This hides the ticks/borders for the plots and makes the empty 6th plot invisible
for ax in axes.flatten():
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3D Thresholding

In [ ]:
import numpy as np
import tifffile
from pathlib import Path
from skimage.filters import threshold_otsu, threshold_local, gaussian
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from skimage.morphology import remove_small_objects
from scipy import ndimage

def segment_3d_objects_robust(file_path: Path, z_start: int, z_end: int, 
                              min_distance=5, 
                              local_block_size=35,
                              anisotropy=(1.0, 1.0, 1.0),
                              noise_floor_fraction=0.5):
    """
    Robust 3D segmentation using Hybrid Thresholding with a relaxed global constraint.
    
    Parameters:
    - noise_floor_fraction: Multiplier for the global Otsu threshold (default 0.5).
                            0.5 means "accept anything brighter than half the Otsu value".
                            This includes dim nuclei while rejecting true background noise.
    """
    # 1. Read and Slice
    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")
    
    full_stack = tifffile.imread(file_path)
    z_start = max(0, z_start)
    z_end = min(full_stack.shape[0], z_end)
    volume_subset = full_stack[z_start:z_end]
    
    print(f"Processing Volume: {volume_subset.shape}")

    # 2. Smoothing
    volume_smooth = gaussian(volume_subset, sigma=0.8, preserve_range=True)

    # 3. Hybrid Thresholding (The Fix)
    print(f"Applying Hybrid Thresholding (Block: {local_block_size}, Floor: {noise_floor_fraction}*Otsu)...")
    
    # A. Relaxed Global Threshold (Noise Floor)
    # Instead of using Otsu as a hard cut, we use it to estimate the signal intensity
    # and set a floor at a fraction of that.
    otsu_val = threshold_otsu(volume_smooth)
    noise_floor = otsu_val * noise_floor_fraction
    mask_global = volume_smooth > noise_floor
    
    # B. Local Threshold (Structure Detection)
    # This finds objects based on local contrast, regardless of absolute brightness.
    adaptive_thresh = threshold_local(volume_smooth, block_size=local_block_size, offset=0)
    mask_local = volume_smooth > adaptive_thresh
    
    # C. Combine
    # We keep pixels that are locally distinct AND above the absolute noise floor.
    binary_mask = mask_global & mask_local
    
    # D. Cleanup (Optional but recommended)
    # Remove tiny specks (e.g., < 20 voxels) that might have survived the lower threshold
    binary_mask = remove_small_objects(binary_mask, min_size=20)

    # 4. Distance Transform & Watershed
    print(f"Computing Distance Transform (Anisotropy: {anisotropy})...")
    distance = ndimage.distance_transform_edt(binary_mask, sampling=anisotropy)
    
    print("Finding Peaks...")
    coords = peak_local_max(
        distance, 
        min_distance=min_distance, 
        labels=binary_mask
    )
    
    mask_peaks = np.zeros(distance.shape, dtype=bool)
    mask_peaks[tuple(coords.T)] = True
    markers, num_features = ndimage.label(mask_peaks)
    
    print(f"Found {num_features} objects. Applying Watershed...")
    labels_3d = watershed(-distance, markers, mask=binary_mask)

    return volume_subset, labels_3d, binary_mask


In [ ]:
from image_processing_tools.util.load_files import find_files_by_pattern
from image_processing_tools.image_class.image_container import ImageContainer
from pathlib import Path
import matplotlib
# matplotlib.use('Agg') # Critical: prevents plotting to screen/RAM

search_path = ("~/A8/Data_Roan/260123_Cocu_Phalloidin/Cocu_Cet112_Ca922_Phalloidin_low_01_CY5, DAPI",)
file_pattern = ("CROP_*",
                "MAX_*",
                "SHARPEST*",
                "C[0-9]*")

config = {
    "preprocessing": {
        "resize_image": False,
        "max_dim": 1080,
        "outlier_percentile": 0.35,
        "quantization": "8bit"
    }
}

# Find the files. The 'files' variable will hold the list of Path objects.
files = find_files_by_pattern(search_path[0], file_pattern[3], verbose=True)

# Create output directory
base_input_dir = Path("~/A8/Data_Roan/").expanduser()
base_output_dir = Path("output/fit_hypha")

# Assuming 'files' is defined in your notebook environment as a list of Path objects
relative_dir = files[0].parent.relative_to(base_input_dir)
current_output_dir = base_output_dir / relative_dir
current_output_dir.mkdir(parents=True, exist_ok=True)

print(f"The current output dir is {current_output_dir}")

spacing = tuple(x/0.065 for x in (0.2,0.065,0.065)) # z, x, y pixel size in micro meters

# Example: Z-step is 1um, XY pixel is 0.5um -> anisotropy=(2, 1, 1)
path_to_file = files[1]

raw, labels, masks = segment_3d_objects_robust(path_to_file, z_start=26, z_end=36, min_distance=4, local_block_size=35, anisotropy=spacing)

In [ ]:
import plotly.graph_objects as go

def visualize_3d_steps(original_stack, binary_mask, labels_stack, filename="3d_steps.html", downsample=2):
    """
    Visualizes Raw Data, Binary Mask, and Segmentation Labels in 3D.
    """
    print(f"Preparing 3D visualization (Downsample: {downsample})...")
    
    # 1. Downsample all inputs
    vol_sub = original_stack[::downsample, ::downsample, ::downsample]
    mask_sub = binary_mask[::downsample, ::downsample, ::downsample].astype(int)
    lab_sub = labels_stack[::downsample, ::downsample, ::downsample]
    
    # 2. Create Coordinate Grid
    z_dim, y_dim, x_dim = vol_sub.shape
    Z, Y, X = np.mgrid[0:z_dim, 0:y_dim, 0:x_dim]
    
    X_flat, Y_flat, Z_flat = X.flatten(), Y.flatten(), Z.flatten()
    
    fig = go.Figure()
    
    # --- Trace 1: Original Image (Gray, Transparent) ---
    vol_flat = vol_sub.flatten()
    vol_min, vol_max = vol_flat.min(), vol_flat.max()
    threshold = vol_min + 0.1 * (vol_max - vol_min)
    
    fig.add_trace(go.Volume(
        x=X_flat, y=Y_flat, z=Z_flat,
        value=vol_flat,
        isomin=threshold, isomax=vol_max,
        opacity=0.1,
        surface_count=15,
        colorscale='Gray',
        name='1. Raw Image',
        hoverinfo='skip'
    ))
    
    # --- Trace 2: Binary Mask (Red, Semi-Transparent) ---
    # We only plot points where mask == 1
    mask_flat = mask_sub.flatten()
    if mask_flat.max() > 0:
        fig.add_trace(go.Volume(
            x=X_flat, y=Y_flat, z=Z_flat,
            value=mask_flat,
            isomin=0.5, isomax=1.0,
            opacity=0.2, # Low opacity to see labels inside if needed
            surface_count=5,
            colorscale=[[0, 'red'], [1, 'red']], # Force single color
            name='2. Binary Mask',
            showscale=False
        ))

    # --- Trace 3: Segmentation Labels (Multi-color, Opaque) ---
    lab_flat = lab_sub.flatten()
    if lab_flat.max() > 0:
        fig.add_trace(go.Volume(
            x=X_flat, y=Y_flat, z=Z_flat,
            value=lab_flat,
            isomin=0.5, isomax=lab_flat.max(),
            opacity=0.6,
            surface_count=10,
            colorscale='Jet',
            name='3. Final Labels',
            colorbar=dict(title='ID', x=1.1)
        ))

    # 3. Layout
    fig.update_layout(
        scene=dict(
            xaxis_title='X', yaxis_title='Y', zaxis_title='Z',
            aspectmode='data'
        ),
        title=f"3D Segmentation Steps (Downsampled {downsample}x)",
        margin=dict(l=0, r=0, b=0, t=40),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
    )
    
    print(f"Saving to {filename}...")
    fig.write_html(filename)
    print("Done.")

# --- Example Usage ---
output_filename = current_output_dir / f"{files[1].stem}_3d_watershed.html"
visualize_3d_steps(raw, masks, labels, filename=output_filename, downsample=8)


In [ ]:
import numpy as np
import plotly.graph_objects as go

def visualize_and_save_3d(original_stack, segmentation_stack, filename="3d_view.html", downsample=2):
    """
    Robust 3D visualization passing full grids to ensure correct rendering.
    """
    print(f"Preparing 3D data (Downsample factor: {downsample})...")
    
    # 1. Downsample
    vol_sub = original_stack[::downsample, ::downsample, ::downsample]
    seg_sub = segmentation_stack[::downsample, ::downsample, ::downsample]
    
    # 2. Create Coordinate Grid
    z_dim, y_dim, x_dim = vol_sub.shape
    Z, Y, X = np.mgrid[0:z_dim, 0:y_dim, 0:x_dim]
    
    # Flatten
    X_flat = X.flatten()
    Y_flat = Y.flatten()
    Z_flat = Z.flatten()
    vol_flat = vol_sub.flatten()
    seg_flat = seg_sub.flatten()
    
    fig = go.Figure()
    
    # --- Trace 1: Original Image ---
    vol_min, vol_max = vol_flat.min(), vol_flat.max()
    # Heuristic: Hide the bottom 10% of intensity to see inside
    threshold = vol_min + 0.1 * (vol_max - vol_min)
    
    fig.add_trace(go.Volume(
        x=X_flat,
        y=Y_flat,
        z=Z_flat,
        value=vol_flat,
        isomin=threshold,
        isomax=vol_max,
        opacity=0.1,       # Very transparent
        surface_count=15,
        colorscale='Gray',
        name='Raw Data',
        hoverinfo='skip'
    ))
    
    # --- Trace 2: Segmentation ---
    # Only add this trace if we actually have objects
    if seg_flat.max() > 0:
        fig.add_trace(go.Volume(
            x=X_flat,
            y=Y_flat,
            z=Z_flat,
            value=seg_flat,
            isomin=0.5,        # Any value < 0.5 (background) is transparent
            isomax=seg_flat.max(),
            opacity=0.6,       # More opaque to stand out
            surface_count=10,  # Fewer surfaces for cleaner look
            colorscale='Jet',
            name='Segmentation',
            colorbar=dict(title='Label ID', x=1.1) # Move colorbar to right
        ))
    else:
        print("WARNING: Segmentation stack is empty (all zeros). No masks will be plotted.")

    # 4. Layout
    fig.update_layout(
        scene=dict(
            xaxis_title='X',
            yaxis_title='Y',
            zaxis_title='Z',
            aspectmode='data'
        ),
        title=f"3D View (Downsampled {downsample}x)",
        margin=dict(l=0, r=0, b=0, t=40)
    )
    
    print(f"Saving to {filename}...")
    fig.write_html(filename)
    print("Done.")

# --- Example Usage ---
# Assuming you have raw_vol and seg_vol from previous steps

output_filename = current_output_dir / f"{files[1].stem}_3d_watershed.html"
visualize_and_save_3d(raw_vol, seg_vol, downsample=8, filename=output_filename)